# 02_sbatch_cosine_runs
run and monitor bge and e5 cosine similarity 

In [1]:
!rm -rf /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/cosines/e5_large/adzuna_month*
!rm -rf /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/cosines/bge_large/adzuna_month*

In [2]:
import subprocess, time, glob
from datetime import datetime
from pathlib import Path

SBATCH = "/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_2_Embeddings_and_Cosine_sim/dev/sbatch/cosine_targets_vs_origin_bge_e5.sbatch"
LOG_GLOB = "/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_2_Embeddings_and_Cosine_sim/dev/sbatch/logs/cos_bge_e5_small_*.out"

print("submit:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
r = subprocess.run(["sbatch", SBATCH], capture_output=True, text=True)
print("stdout:", r.stdout.strip() or "<empty>")
print("stderr:", r.stderr.strip() or "<empty>")
r.check_returncode()

job_id = r.stdout.strip().split()[-1]
print("job_id:", job_id)

def squeue(job_id: str) -> str:
    return subprocess.run(["squeue", "-j", job_id], capture_output=True, text=True).stdout

# wait until job appears
for _ in range(60):
    q = squeue(job_id)
    if job_id in q:
        break
    time.sleep(2)

print("queue:\n", q.strip() or "<not in queue>")

print("\nTailing latest array-task log (Ctrl+C to stop):")
try:
    while True:
        outs = sorted(glob.glob("/projects/a5u/adu_dev/aisi-economy-index/logs/*.out"))
        job_outs = [p for p in outs if f"_{job_id}_" in Path(p).name]
        if job_outs:
            p = job_outs[-1]
            tail = subprocess.run(["tail", "-n", "60", p], capture_output=True, text=True).stdout
            print("\n---", Path(p).name, datetime.now().strftime("%H:%M:%S"), "---\n", tail.rstrip())
        else:
            print("waiting for log file...")

        q = squeue(job_id)
        if job_id not in q:
            print("\njob finished:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
            break
        time.sleep(10)
except KeyboardInterrupt:
    print("\nstopped tailing logs")

# quick verify: count cosine part files written so far
out_bge = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/cosines/bge_large")
out_e5  = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/cosines/e5_large")

bge_parts = list(out_bge.rglob("part_*.npz"))
e5_parts  = list(out_e5.rglob("part_*.npz"))

print("\nOutputs so far:")
print("  bge parts:", len(bge_parts))
print("  e5  parts:", len(e5_parts))

# quick inspect one output file if present
def peek_one(parts, label):
    if not parts:
        print(f"  {label}: no parts yet")
        return
    p = sorted(parts)[0]
    d = np.load(p, allow_pickle=True)
    print(f"\n{label} sample file:", p)
    print("  keys:", d.files)
    for k in ["job_ids", "role_topk_idx", "role_topk_val", "task_topk_idx", "task_topk_val", "chunk_start", "chunk_end"]:
        if k in d.files:
            print(" ", k, d[k].shape, d[k].dtype)

import numpy as np
peek_one(bge_parts, "BGE")
peek_one(e5_parts, "E5")


submit: 2026-01-31 20:00:20
stdout: Submitted batch job 2093747
stderr: <empty>
job_id: 2093747
queue:
 JOBID         USER PARTITION                     NAME ST TIME_LIMIT       TIME  TIME_LEFT NODES NODELIST(REASON)
2093747 autonomyluiz     workq         cos_bge_e5_small PD    6:00:00       0:00    6:00:00     1 (None)

Tailing latest array-task log (Ctrl+C to stop):
waiting for log file...

--- cos_bge_e5_small_2093747_9.out 20:00:31 ---
 TASK_ID: 9
MONTH: adzuna_month05 SHARD: 1
BGE_IN: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/embeddings/run_embed_target/bge_large/adzuna_month05/adzuna_month05_shard01_of02.npz exists= True
E5_IN : /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/embeddings/run_embed_target/e5_large/adzuna_month05/adzuna_month05_shard01_of02.npz exists= True
cuda: True
gpu: NVIDIA GH200 120GB
[SETUP] device=cuda dtype=torch.float16 topk=

In [3]:
import hashlib
from pathlib import Path

month="adzuna_month05"
shard="01"

bge = Path(f"/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/embeddings/run_embed_target/bge_large/{month}/{month}_shard{shard}_of02.npz")
e5  = Path(f"/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/embeddings/run_embed_target/e5_large/{month}/{month}_shard{shard}_of02.npz")

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

print("BGE:", bge.exists(), bge)
print("E5 :", e5.exists(), e5)

hb = sha256(bge)
he = sha256(e5)

print("\nsha256 BGE:", hb)
print("sha256 E5 :", he)
print("same bytes:", hb == he)


BGE: True /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/embeddings/run_embed_target/bge_large/adzuna_month05/adzuna_month05_shard01_of02.npz
E5 : True /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/embeddings/run_embed_target/e5_large/adzuna_month05/adzuna_month05_shard01_of02.npz

sha256 BGE: a41f4ce6c5525ea58884951730810bb9932834ea0a38aff20ff2c6d393935112
sha256 E5 : 8fb6b7adaba3835ee4b1f62b76b4902a3a7a99d2ca7e68eb29795873b6aa7176
same bytes: False


In [4]:
import numpy as np

# Absolute paths for the specific month/shard from your logs
e5_path = "/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/cosines/e5_large/adzuna_month05_shard01/part_0000.npz"
bge_path = "/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/cosines/bge_large/adzuna_month05_shard01/part_0000.npz"

# Load the new parts
e5_test = np.load(e5_path)
bge_test = np.load(bge_path)

# Check the first few role matches
print("=== Validation Test (Month 05, Shard 01) ===")
print("E5 Top Match Indices (first job): ", e5_test['role_topk_idx'][0])
print("BGE Top Match Indices (first job):", bge_test['role_topk_idx'][0])

# Calculate local Jaccard for this first row
set_e5 = set(e5_test['role_topk_idx'][0])
set_bge = set(bge_test['role_topk_idx'][0])
intersection = len(set_e5.intersection(set_bge))
union = len(set_e5.union(set_bge))
print(f"Local Jaccard Overlap: {intersection/union:.2f}")

=== Validation Test (Month 05, Shard 01) ===
E5 Top Match Indices (first job):  [416 796 261 440 434]
BGE Top Match Indices (first job): [351 550   3 339 326]
Local Jaccard Overlap: 0.00


In [5]:
import pandas as pd
# Load your ONET metadata
df_onet = pd.read_csv("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/embeddings/standard_df_onet_occupations_description_activities_and_tasks.csv")
# Filter it exactly like your script did (dropping empty roles)
df_onet = df_onet[df_onet["Job Role Description"].fillna("").str.strip() != ""].reset_index(drop=True)

# Look at the top match for each
print("E5 Top Match Text:", df_onet.iloc[416]["Job Role Description"])
print("BGE Top Match Text:", df_onet.iloc[351]["Job Role Description"])

E5 Top Match Text: Orthoptists - Diagnose and treat visual system disorders such as binocular vision and eye movement impairments.
BGE Top Match Text: Public Relations Specialists - Promote or create an intended public image for individuals, groups, or organizations. May write or select material for release to various communications media. May specialize in using social media.


In [6]:
import numpy as np

path = "/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/embeddings/run_embed_onet/onet_row_idx_e5.npy"

arr = np.load(path, allow_pickle=True)
print(type(arr))
print(arr.shape if hasattr(arr, "shape") else "no shape")
print(arr[:10])


<class 'numpy.ndarray'>
(894,)
[0 1 2 3 4 5 6 7 8 9]


In [9]:
import numpy as np

e5_top = np.array([416, 796, 261, 440, 434])
bge_top = np.array([351, 550,   3, 339, 326])

rowmap = np.load("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/embeddings/run_embed_onet/onet_row_idx_e5.npy")

# If rowmap length equals number of E5 rows, this should map E5-local idx -> global onet row idx
e5_top_global = rowmap[e5_top]

print("E5 mapped to global:", e5_top_global.tolist())
print("BGE global:", bge_top.tolist())

overlap = len(set(e5_top_global.tolist()) & set(bge_top.tolist())) / len(set(e5_top_global.tolist()) | set(bge_top.tolist()))
print("Jaccard after mapping:", overlap)


E5 mapped to global: [416, 796, 261, 440, 434]
BGE global: [351, 550, 3, 339, 326]
Jaccard after mapping: 0.0


In [12]:
import numpy as np

p = "/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/embeddings/aspectt_vectors.npz"
z = np.load(p, allow_pickle=True)

print("keys:", z.files)
for k in z.files:
    a = z[k]
    print(k, type(a), getattr(a, "dtype", None), getattr(a, "shape", None))


keys: ['titles', 'columns', 'levels', 'importance']
titles <class 'numpy.ndarray'> object (894,)
columns <class 'numpy.ndarray'> object (161,)
levels <class 'numpy.ndarray'> float32 (894, 161)
importance <class 'numpy.ndarray'> float32 (894, 161)


In [13]:
import numpy as np

z = np.load("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/embeddings/aspectt_vectors.npz", allow_pickle=True)
titles = z["titles"]

e5_top_global = np.array([416, 796, 261, 440, 434])
bge_top = np.array([351, 550,   3, 339, 326])

print("E5 titles:")
for i in e5_top_global:
    print(i, titles[i])

print("\nBGE titles:")
for i in bge_top:
    print(i, titles[i])


E5 titles:
416 Orthoptists
796 Furniture Finishers
261 Arbitrators, Mediators, and Conciliators
440 Ophthalmic Medical Technologists
434 Ophthalmic Medical Technicians

BGE titles:
351 Public Relations Specialists
550 Advertising Sales Agents
3 Advertising and Promotions Managers
339 Media Programming Directors
326 Art Directors


In [14]:
i = 0
print("job_id:", job_ids[i])
print("domain:", domains[i] if "domains" in globals() else None)
print("short_desc:", short_desc[i][:200] if "short_desc" in globals() else None)
print("tasks:", tasks[i][:200] if "tasks" in globals() else None)


NameError: name 'job_ids' is not defined